# Candle Prediction using Market Depth

In [61]:
from pathlib import Path
import pandas as pd
import ast
import numpy as np
import datetime
from datetime import timedelta
from utils import resample_fractional_minute

In [62]:
# ---- Input ------
date_ = "20APR2026"
file_name = "NIFTY2642124300PE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)

In [63]:
file_name

'NIFTY2642124300PE.xlsx'

In [64]:
# ---- Input ------
N = 6

# M1 and M3 inputs
IMBALANCE_UPPER = 2.5
IMBALANCE_LOWER = 0.4

LTP_LOOKBACK_SEC = 3          # reduced
MIN_TICK_VELOCITY = 2        # increased # it means atleast 8 ticks per second should be there

LAMBDA = 0.7                 # faster decay
THRESHOLD = 0.45             # stricter

PRICE_MOVE_THRESHOLD = 0.8   # tune based on instrument
#------------------

In [65]:
100 +PRICE_MOVE_THRESHOLD

100.8

In [66]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode'],
      dtype='str')

In [67]:
df.head(2)

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,total_buy_quantity,total_sell_quantity,ohlc,change,oi,oi_day_high,oi_day_low,depth,tradable,mode
0,16238338,NIFTY2642124300PE,NaN,2026-04-20 09:10:06.815,2026-04-17 15:29:59,134.55,65,0.00,PE,atm_minus_2.0,...,0,0,"{'open': 269.9, 'high': 306.45, 'low': 123.0, ...",0.000000,4418375,4418375,4418375,"{'buy': [{'quantity': 0, 'price': 0.0, 'orders...",True,full
1,16238338,NIFTY2642124300PE,NaN,2026-04-20 09:15:00.562,2026-04-20 09:15:00,98.00,65,117.08,PE,atm_minus_2.0,...,37245,65585,"{'open': 121.0, 'high': 151.0, 'low': 101.05, ...",-27.164623,4418375,4418375,4418375,"{'buy': [{'quantity': 325, 'price': 111.3, 'or...",True,full


In [68]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-20 09:10:06.815000
2026-04-17 15:29:59


In [69]:
# M3 Signal
def add_m3_signal(df):
    return np.where(
        df["bucket_time"] == df["minute"],
        np.where(
            df["open"] < df["close"], "BUY",
            np.where(df["open"] > df["close"], "SELL", None)
        ),
        None
    )

In [70]:
clubbed_df = resample_fractional_minute(df, "last_trade_time", N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["m3_signal"] = add_m3_signal(clubbed_df)

In [71]:
clubbed_df.head(5)

,bucket_time,minute,open,high,low,close,candle_type,minute_open,minute_high,minute_low,minute_close,bucket_time_next,m3_signal
0,2026-04-17 15:29:50,2026-04-17 15:29:00,134.55,134.55,134.55,134.55,SELL,134.55,134.55,134.55,134.55,2026-04-20 09:15:00,NaN
1,2026-04-20 09:15:00,2026-04-20 09:15:00,98.00,131.50,92.45,131.50,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:10,BUY
2,2026-04-20 09:15:10,2026-04-20 09:15:00,126.75,133.40,123.45,128.35,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:20,NaN
3,2026-04-20 09:15:20,2026-04-20 09:15:00,125.55,141.00,125.55,140.50,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:30,NaN
4,2026-04-20 09:15:30,2026-04-20 09:15:00,140.40,148.85,138.60,148.85,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:40,NaN


In [72]:
def extract_features(tick_row, ltp_series, current_time, bucket_tick_df):
    data = ast.literal_eval(tick_row["depth"])

    # --- imbalance ---
    bid_sum = sum([x["quantity"] for x in data["buy"]])
    ask_sum = sum([x["quantity"] for x in data["sell"]])

    if ask_sum == 0:
        return None

    imbalance_ratio = bid_sum / ask_sum

    # --- best bid/ask ---
    best_bid = data["buy"][0]["price"]
    best_ask = data["sell"][0]["price"]

    # --- ltp change ---
    lookback_time = current_time - timedelta(seconds=LTP_LOOKBACK_SEC)
    past_ticks = ltp_series[ltp_series.index <= lookback_time]

    if len(past_ticks) == 0:
        return None

    ltp_5s_ago = past_ticks.iloc[-1]
    ltp_change = tick_row["last_price"] - ltp_5s_ago

    # --- tick velocity ---
    window_1s_start = current_time - timedelta(seconds=1)
    ticks_last_1s = bucket_tick_df[
        (bucket_tick_df["last_trade_time"] >= window_1s_start) &
        (bucket_tick_df["last_trade_time"] <= current_time)
    ]

    tick_velocity = len(ticks_last_1s)

    return {
        "imbalance_ratio": imbalance_ratio,
        "ltp_change": ltp_change,
        "tick_velocity": tick_velocity,
        "best_bid": best_bid,
        "best_ask": best_ask
    }

In [73]:
# def compute_m1_score(features):
#     score = 0
#
#     if features["imbalance_ratio"] >= IMBALANCE_UPPER:
#         score += 1
#     elif features["imbalance_ratio"] <= IMBALANCE_LOWER:
#         score -= 1
#
#     # if features["ltp_change"] > 0:
#     #     score += 1
#     # elif features["ltp_change"] < 0:
#     #     score -= 1
#
#     price_move = features["ltp_change"]
#
#     if price_move > PRICE_MOVE_THRESHOLD:
#         score += 1   # strong bullish move
#     elif price_move > 0:
#         score += 1   # weak bullish
#     elif price_move < -PRICE_MOVE_THRESHOLD:
#         score -= 1   # strong bearish
#     elif price_move < 0:
#         score -= 1   # weak bearish
#
#     if features["tick_velocity"] >= MIN_TICK_VELOCITY:
#         if features["ltp_change"] > 0:
#             score += 1
#         elif features["ltp_change"] < 0:
#             score -= 1
#
#     return score

# improved by chatGPT
# def compute_m1_score(features):
#
#     imbalance = features["imbalance_ratio"]
#     price_move = features["ltp_change"]
#     velocity = features["tick_velocity"]
#
#     # ---------------------------
#     # 1. Imbalance normalization
#     # ---------------------------
#     imbalance_score = (imbalance - 1) / (imbalance + 1)
#     imbalance_score = np.clip(imbalance_score, -1, 1)
#
#     # ---------------------------
#     # 2. Price movement normalization
#     # ---------------------------
#     # scale factor (IMPORTANT: tune per instrument)
#     K = 200
#
#     price_score = price_move * K
#     price_score = np.clip(price_score, -1, 1)
#
#     # ---------------------------
#     # 3. Velocity normalization
#     # ---------------------------
#     if velocity >= MIN_TICK_VELOCITY:
#         velocity_score = (velocity - MIN_TICK_VELOCITY) / MIN_TICK_VELOCITY
#         velocity_score = np.clip(velocity_score, 0, 1)
#     else:
#         velocity_score = 0
#
#     # ---------------------------
#     # 4. Combine (weighted)
#     # ---------------------------
#     w1 = 0.3   # imbalance
#     w2 = 0.5   # price movement (most important)
#     w3 = 0.2   # velocity
#
#     score = (
#         w1 * imbalance_score +
#         w2 * price_score +
#         w3 * velocity_score * np.sign(price_score)
#     )
#
#     return score

# improved by Claude
def compute_m1_score(features, atr_estimate=5.0):
    """
    atr_estimate: rolling ATR of LTP over recent buckets (pass in dynamically).
                  Start with 5.0 for Nifty options and update each bucket.
    """
    imbalance = features["imbalance_ratio"]
    price_move = features["ltp_change"]
    velocity   = features["tick_velocity"]

    # -------------------------------------------------------
    # 1. Imbalance: log-ratio instead of (x-1)/(x+1)
    #    - More sensitive near 1.0, still bounded
    #    - Asymmetric clip: sellers in options matter more
    # -------------------------------------------------------
    log_imbalance = np.log(imbalance + 1e-6)
    imbalance_score = np.clip(log_imbalance / np.log(10), -1, 1)
    # Slightly penalise pure ask-side dominance (options: ask lifting = bullish)
    if imbalance_score < 0:
        imbalance_score *= 0.8  # soften sell-side imbalance signal

    # -------------------------------------------------------
    # 2. Price score: normalise by ATR instead of fixed K
    #    - atr_estimate should be updated each candle bucket
    # -------------------------------------------------------
    price_score = price_move / (atr_estimate + 1e-6)
    price_score = np.clip(price_score, -1, 1)

    # -------------------------------------------------------
    # 3. Velocity: treat as multiplier, not additive component
    #    - Only acts as confidence gate, not direction
    #    - Use soft sigmoid instead of hard threshold
    # -------------------------------------------------------
    # Soft gate: 0 below MIN_TICK_VELOCITY, rises smoothly above
    velocity_gate = 1 / (1 + np.exp(-(velocity - MIN_TICK_VELOCITY) / 3))
    # Normalize to [0, 1] range starting from 0.5 at threshold
    velocity_gate = (velocity_gate - 0.5) * 2
    velocity_gate = np.clip(velocity_gate, 0, 1)

    # -------------------------------------------------------
    # 4. Combine: velocity gates the combined score
    #    instead of adding direction-amplified noise
    # -------------------------------------------------------
    w1, w2 = 0.35, 0.65
    raw_score = w1 * imbalance_score + w2 * price_score

    # Velocity multiplier: low velocity shrinks signal toward zero
    # min_velocity_factor: even with 0 velocity, don't zero out completely
    min_velocity_factor = 0.3
    velocity_factor = min_velocity_factor + (1 - min_velocity_factor) * velocity_gate

    score = raw_score * velocity_factor

    return score

In [74]:
# def compute_m2_score(features, prev_bid, prev_ask):
#     if prev_bid is None or prev_ask is None:
#         return None   # IMPORTANT: no score for first state
#
#     if features["best_bid"] > prev_bid and features["best_ask"] > prev_ask:
#         return 1
#     elif features["best_bid"] < prev_bid and features["best_ask"] < prev_ask:
#         return -1
#     else:
#         return 0

# improved by chatGPT
# def compute_m2_score(features, prev_bid, prev_ask):
#
#     if prev_bid is None or prev_ask is None:
#         return None
#
#     bid = features["best_bid"]
#     ask = features["best_ask"]
#
#     if bid > prev_bid and ask > prev_ask:
#         return 1.0
#     elif bid < prev_bid and ask < prev_ask:
#         return -1.0
#     else:
#         return 0.0

# imrpoved by claude
def compute_m2_score(features, prev_bid, prev_ask, mid_price_ref=None):
    """
    mid_price_ref: recent LTP or mid-price, used to normalise move magnitude.
    """
    if prev_bid is None or prev_ask is None:
        return None

    bid = features["best_bid"]
    ask = features["best_ask"]

    bid_delta = bid - prev_bid
    ask_delta = ask - prev_ask

    # -------------------------------------------------------
    # 1. Compute separate bid and ask direction scores
    # -------------------------------------------------------
    # Normalise by tick size (0.05 for Nifty options)
    TICK_SIZE = 0.05
    bid_ticks = bid_delta / TICK_SIZE
    ask_ticks = ask_delta / TICK_SIZE

    # Soft-clip: compress large moves but don't saturate
    bid_score = np.tanh(bid_ticks / 3)
    ask_score = np.tanh(ask_ticks / 3)

    # -------------------------------------------------------
    # 2. Agreement-weighted combination
    #    - Both moving same direction → strong signal
    #    - Only one moving → partial signal (not zero)
    #    - Moving opposite directions → noisy, reduce weight
    # -------------------------------------------------------
    if np.sign(bid_ticks) == np.sign(ask_ticks) and bid_ticks != 0:
        # Both agree → full combined signal
        m2_score = (bid_score + ask_score) / 2
        agreement_factor = 1.0
    elif bid_ticks == 0 and ask_ticks != 0:
        # Only ask moved → partial, slightly less reliable
        m2_score = ask_score * 0.6
        agreement_factor = 0.6
    elif ask_ticks == 0 and bid_ticks != 0:
        # Only bid moved → partial
        m2_score = bid_score * 0.6
        agreement_factor = 0.6
    else:
        # Opposite directions → book spreading/narrowing, reduce heavily
        m2_score = (bid_score + ask_score) / 2
        agreement_factor = 0.3

    m2_score = np.clip(m2_score, -1, 1)
    return m2_score

In [75]:
# def aggregate_scores(scores, min_count=5):
#     if len(scores) < min_count:
#         return None, "NO_SIGNAL"
#
#     scores = np.array(scores)
#     N = len(scores)
#
#     weights = np.exp(-LAMBDA * (np.arange(N)[::-1]))
#     weighted_sum = np.sum(scores * weights)
#     total_weight = np.sum(weights)
#
#     final_score = weighted_sum / total_weight
#
#     pos_ratio = np.sum(scores > 0) / N
#     neg_ratio = np.sum(scores < 0) / N
#
#     if pos_ratio < 0.6 and neg_ratio < 0.6:
#         return final_score, "NO_SIGNAL"
#
#     if final_score > THRESHOLD:
#         return final_score, "BUY"
#     elif final_score < -THRESHOLD:
#         return final_score, "SELL"
#     else:
#         return final_score, "NO_SIGNAL"

# improved by chatGPT
# def aggregate_scores(scores, min_count=5):
#
#     if len(scores) < min_count:
#         return None, "NO_SIGNAL"
#
#     scores = np.array(scores)
#     N = len(scores)
#
#     # ---------------------------
#     # Exponential weighting
#     # ---------------------------
#     weights = np.exp(-LAMBDA * (np.arange(N)[::-1]))
#     weighted_sum = np.sum(scores * weights)
#     total_weight = np.sum(weights)
#
#     final_score = weighted_sum / total_weight
#
#     # ---------------------------
#     # Stability filter
#     # ---------------------------
#     std_dev = np.std(scores)
#
#     if std_dev > 0.8:   # noisy signals
#         return final_score, "NO_SIGNAL"
#
#     # ---------------------------
#     # Decision
#     # ---------------------------
#     if final_score > THRESHOLD:
#         return final_score, "BUY"
#     elif final_score < -THRESHOLD:
#         return final_score, "SELL"
#     else:
#         return final_score, "NO_SIGNAL"

# Improved by claude
def aggregate_scores(scores, min_count=5, label="M1"):

    if len(scores) < min_count:
        return None, "NO_SIGNAL"

    scores = np.array(scores)
    N = len(scores)

    # -------------------------------------------------------
    # 1. Separate noise filter from directional filter
    # -------------------------------------------------------
    std_dev = np.std(scores)
    mean_abs = np.mean(np.abs(scores))

    # Noise ratio: high std relative to mean signal = noise
    noise_ratio = std_dev / (mean_abs + 1e-6)

    # -------------------------------------------------------
    # 2. Adaptive lambda: slow decay when signals are consistent
    # -------------------------------------------------------
    if noise_ratio < 1.0:
        # Consistent signals: use slower decay to preserve history
        effective_lambda = LAMBDA * 0.6
    else:
        # Noisy: use faster decay, trust only recent signals
        effective_lambda = LAMBDA * 1.3

    weights = np.exp(-effective_lambda * (np.arange(N)[::-1]))
    weighted_sum = np.sum(scores * weights)
    total_weight  = np.sum(weights)
    final_score   = weighted_sum / total_weight

    # -------------------------------------------------------
    # 3. Directional consistency check
    #    (replaces blunt std_dev > 0.8 cutoff)
    # -------------------------------------------------------
    positive_frac = np.mean(scores > 0)
    negative_frac = np.mean(scores < 0)
    # If more than 35% of signals disagree with the majority → uncertain
    minority_frac = min(positive_frac, negative_frac)

    if minority_frac > 0.35:
        return final_score, "NO_SIGNAL"

    # -------------------------------------------------------
    # 4. Score threshold check
    # -------------------------------------------------------
    if final_score > THRESHOLD:
        return final_score, "BUY"
    elif final_score < -THRESHOLD:
        return final_score, "SELL"
    else:
        return final_score, "NO_SIGNAL"

In [76]:
def get_strategy_scores(df, df_grp_4):

    for idx, row in df_grp_4.iterrows():

        # process only first quarter
        if idx % 4 != 0:
            continue

        bucket_time = row["bucket_time"]
        bucket_time_next = row["bucket_time_next"]

        bucket_tick_df = df[
            (df["last_trade_time"] >= bucket_time) &
            (df["last_trade_time"] < bucket_time_next)
        ].copy().sort_values("last_trade_time")

        if len(bucket_tick_df) == 0:
            continue

        ltp_series = bucket_tick_df.set_index("last_trade_time")["last_price"]

        # --- score storage ---
        m1_scores = []
        m2_scores = []

        # --- state tracking ---
        prev_bid = None
        prev_ask = None
        prev_depth = None
        prev_ltp = None

        # Before the tick loop, compute ATR estimate for the bucket
        price_range = bucket_tick_df["last_price"].max() - bucket_tick_df["last_price"].min()
        atr_estimate = max(price_range, 1.0)  # floor at 1.0 to avoid division issues
        # Pass atr_estimate into compute_m1_score(features, atr_estimate)

        # =========================
        # TICK LOOP
        # =========================
        for _, tick_row in bucket_tick_df.iterrows():

            current_time = tick_row["last_trade_time"]
            current_depth = tick_row["depth"]
            current_ltp = tick_row["last_price"]

            # -----------------------------------
            # DUPLICATE FILTER (Model 1 fix)
            # -----------------------------------
            if prev_depth is not None and prev_ltp is not None:
                if current_depth == prev_depth and current_ltp == prev_ltp:
                    continue

            # -----------------------------------
            # FEATURE EXTRACTION
            # -----------------------------------
            features = extract_features(
                tick_row, ltp_series, current_time, bucket_tick_df
            )

            if features is None:
                prev_depth = current_depth
                prev_ltp = current_ltp
                continue

            # -----------------------------------
            # MODEL 1
            # -----------------------------------
            m1_score = compute_m1_score(features)
            m1_scores.append(m1_score)

            # -----------------------------------
            # MODEL 2 (depth-change driven)
            # -----------------------------------
            best_bid = features["best_bid"]
            best_ask = features["best_ask"]

            depth_changed = False

            if prev_bid is None or prev_ask is None:
                depth_changed = True
            elif best_bid != prev_bid or best_ask != prev_ask:
                depth_changed = True

            if depth_changed:
                m2_score = compute_m2_score(features, prev_bid, prev_ask)

                if m2_score is not None:
                    m2_scores.append(m2_score)

                prev_bid = best_bid
                prev_ask = best_ask

            # update duplicate tracking
            prev_depth = current_depth
            prev_ltp = current_ltp

        # =========================
        # AGGREGATION
        # =========================
        m1_final, m1_signal = aggregate_scores(m1_scores, min_count=5)
        m2_final, m2_signal = aggregate_scores(m2_scores, min_count=3)

        if m1_final is not None and m2_final is not None:
            m1_m2_score = m1_final * (1 + 0.4 * m2_final)
        else:
            m1_m2_score = m1_final

        # =========================
        # STORE RESULTS
        # =========================
        df_grp_4.loc[idx, "m1_score"] = m1_final
        df_grp_4.loc[idx, "m1_signal"] = m1_signal

        df_grp_4.loc[idx, "m2_score"] = m2_final
        df_grp_4.loc[idx, "m2_signal"] = m2_signal

        df_grp_4.loc[idx, "m1_m2_score"] = m1_m2_score
    return df_grp_4

In [77]:
1

1

In [78]:
%%time
# inputs
df_grp_4 = get_strategy_scores(df, clubbed_df)

CPU times: total: 9.11 s
Wall time: 9.44 s


In [80]:
df_grp_4.head(20)

,bucket_time,minute,open,high,low,close,candle_type,minute_open,minute_high,minute_low,minute_close,bucket_time_next,m3_signal,m1_score,m1_signal,m2_score,m2_signal,m1_m2_score
0,2026-04-17 15:29:50,2026-04-17 15:29:00,134.55,134.55,134.55,134.55,SELL,134.55,134.55,134.55,134.55,2026-04-20 09:15:00,NaN,None,NO_SIGNAL,None,NO_SIGNAL,None
1,2026-04-20 09:15:00,2026-04-20 09:15:00,98.00,131.50,92.45,131.50,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:10,BUY,NaN,NaN,NaN,NaN,NaN
2,2026-04-20 09:15:10,2026-04-20 09:15:00,126.75,133.40,123.45,128.35,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:20,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-20 09:15:20,2026-04-20 09:15:00,125.55,141.00,125.55,140.50,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:30,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-20 09:15:30,2026-04-20 09:15:00,140.40,148.85,138.60,148.85,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:40,NaN,0.034866,NO_SIGNAL,0.370873,NO_SIGNAL,0.040039
5,2026-04-20 09:15:40,2026-04-20 09:15:00,150.65,154.85,148.80,154.00,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:50,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-04-20 09:15:50,2026-04-20 09:15:00,156.65,156.65,147.40,148.40,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:16:00,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-04-20 09:16:00,2026-04-20 09:16:00,142.90,146.05,138.95,145.95,BUY,142.90,162.85,138.95,151.50,2026-04-20 09:16:10,BUY,NaN,NaN,NaN,NaN,NaN
8,2026-04-20 09:16:10,2026-04-20 09:16:00,148.00,156.20,148.00,154.40,BUY,142.90,162.85,138.95,151.50,2026-04-20 09:16:20,NaN,-0.073679,NO_SIGNAL,0.584172,BUY,-0.090895
9,2026-04-20 09:16:20,2026-04-20 09:16:00,151.80,156.05,151.20,155.70,BUY,142.90,162.85,138.95,151.50,2026-04-20 09:16:30,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
# predictions

In [53]:
import numpy as np

# M1
# df_grp_4["predicted"] = df_grp_4["m1_signal"]

# # M2
# df_grp_4["predicted"] = df_grp_4["m2_signal"]
# #
# M3
df_grp_4["predicted"] = df_grp_4["m3_signal"]
# #
# #
# # # M1 and M2
# df_grp_4["predicted"] = df_grp_4["m1_signal"].where(
#     df_grp_4["m1_signal"] == df_grp_4["m2_signal"]
# )
# #
# # # M1 and M3
# df_grp_4["predicted"] = df_grp_4["m1_signal"].where(
#     df_grp_4["m1_signal"] == df_grp_4["m3_signal"]
# )
# #
# # M2 and M3
# df_grp_4["predicted"] = df_grp_4["m2_signal"].where(
#     df_grp_4["m2_signal"] == df_grp_4["m3_signal"]
# )
# # #
# # M1, M2, M3
# df_grp_4["predicted"] = df_grp_4["m1_signal"].where(
#     (df_grp_4["m1_signal"].notna()) &
#     (df_grp_4["m1_signal"] == df_grp_4["m2_signal"]) &
#     (df_grp_4["m1_signal"] == df_grp_4["m3_signal"])
# )


In [54]:
df_grp_4["predicted"].value_counts()

predicted
SELL    190
BUY     174
Name: count, dtype: int64

In [55]:
df_grp_4[df_grp_4["predicted"].isin(["BUY", "SELL"])].head()

,bucket_time,minute,open,high,low,close,candle_type,minute_open,minute_high,minute_low,minute_close,bucket_time_next,m3_signal,m1_score,m1_signal,m2_score,m2_signal,m1_m2_score,predicted
1,2026-04-20 09:15:00,2026-04-20 09:15:00,98.00,133.4,92.45,132.00,BUY,98.00,156.65,92.45,148.4,2026-04-20 09:15:15,BUY,NaN,NaN,NaN,NaN,NaN,BUY
5,2026-04-20 09:16:00,2026-04-20 09:16:00,142.90,156.0,138.95,154.70,BUY,142.90,162.85,138.95,151.5,2026-04-20 09:16:15,BUY,NaN,NaN,NaN,NaN,NaN,BUY
9,2026-04-20 09:17:00,2026-04-20 09:17:00,150.25,157.6,147.75,148.65,SELL,150.25,157.60,135.55,135.7,2026-04-20 09:17:15,SELL,NaN,NaN,NaN,NaN,NaN,SELL
13,2026-04-20 09:18:00,2026-04-20 09:18:00,139.60,139.6,134.60,137.25,BUY,139.60,144.60,130.90,141.2,2026-04-20 09:18:15,SELL,NaN,NaN,NaN,NaN,NaN,SELL
17,2026-04-20 09:19:00,2026-04-20 09:19:00,141.80,144.6,138.00,138.95,SELL,141.80,144.60,124.35,126.2,2026-04-20 09:19:15,SELL,NaN,NaN,NaN,NaN,NaN,SELL


In [56]:
from sklearn.metrics import confusion_matrix, classification_report

# Optional: remove rows without prediction
eval_df = df_grp_4.dropna(subset=["predicted", "candle_type"]).copy()

# If you want to ignore NO_SIGNAL:
eval_df = eval_df[eval_df["predicted"] != "NO_SIGNAL"]

y_true = eval_df["candle_type"]
y_pred = eval_df["predicted"]

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=["BUY", "SELL"])

print("Confusion Matrix (rows=true, cols=pred):")
print(cm)

# Detailed report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Precision : When the model says "BUY," how often is it actually a buy?
# Recall: Of all the actual "BUY" opportunities that happened, how many did the model catch?


Confusion Matrix (rows=true, cols=pred):
[[117  55]
 [ 57 135]]

Classification Report:
              precision    recall  f1-score   support

         BUY       0.67      0.68      0.68       172
        SELL       0.71      0.70      0.71       192

    accuracy                           0.69       364
   macro avg       0.69      0.69      0.69       364
weighted avg       0.69      0.69      0.69       364



In [57]:
# Filter BUY predictions
buy_preds = eval_df[eval_df["predicted"] == "BUY"]

# Counts
correct_buy = (buy_preds["candle_type"] == "BUY").sum()
incorrect_buy = (buy_preds["candle_type"] == "SELL").sum()

total_buy_preds = len(buy_preds)

# Precision (BUY accuracy relative to BUY predictions)
buy_precision = correct_buy / total_buy_preds if total_buy_preds > 0 else 0

print("Total BUY Predictions:", total_buy_preds)
print("Correct BUY Predictions:", correct_buy)
print("Incorrect BUY Predictions:", incorrect_buy)
print("BUY Precision:", buy_precision)

Total BUY Predictions: 174
Correct BUY Predictions: 117
Incorrect BUY Predictions: 57
BUY Precision: 0.6724137931034483


In [58]:
# Filter SELL predictions
sell_preds = eval_df[eval_df["predicted"] == "SELL"]

# Counts
correct_sell = (sell_preds["candle_type"] == "SELL").sum()
incorrect_sell = (sell_preds["candle_type"] == "BUY").sum()

total_sell_preds = len(sell_preds)

# Precision (BUY accuracy relative to BUY predictions)
sell_precision = correct_sell / total_sell_preds if total_sell_preds > 0 else 0

print("Total SELL Predictions:", total_sell_preds)
print("Correct SELL Predictions:", correct_sell)
print("Incorrect SELL Predictions:", incorrect_sell)
print("SELL Precision:", sell_precision)

Total SELL Predictions: 190
Correct SELL Predictions: 135
Incorrect SELL Predictions: 55
SELL Precision: 0.7105263157894737


In [59]:
correct = (eval_df["candle_type"] == eval_df["predicted"]).sum()
incorrect = (eval_df["candle_type"] != eval_df["predicted"]).sum()

total = len(eval_df)

print(f"Total evaluated: {total}")
print(f"Correct predictions: {correct}")
print(f"Incorrect predictions: {incorrect}")
print(f"Accuracy: {correct / total:.4f}")

Total evaluated: 364
Correct predictions: 252
Incorrect predictions: 112
Accuracy: 0.6923


In [60]:
1

1

In [97]:
# df_grp_4[df_grp_4["predicted"].isin(["BUY", "SELL"])].to_csv("df_grp_4.csv", index=False)

In [98]:
df_grp_4[df_grp_4["predicted"].isin(["BUY"])]

,bucket_time,minute,open,high,low,close,candle_type,minute_open,minute_high,minute_low,minute_close,bucket_time_next,m3_signal,m1_score,m1_signal,m2_score,m2_signal,m1_m2_score,predicted
1,2026-04-20 09:15:00,2026-04-20 09:15:00,98.00,131.50,92.45,124.10,BUY,98.00,156.65,92.45,148.40,2026-04-20 09:15:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY
6,2026-04-20 09:16:00,2026-04-20 09:16:00,142.90,152.95,138.95,152.95,BUY,142.90,162.85,138.95,151.50,2026-04-20 09:16:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY
31,2026-04-20 09:21:00,2026-04-20 09:21:00,114.40,118.35,113.40,115.00,BUY,114.40,120.90,113.40,114.95,2026-04-20 09:21:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY
41,2026-04-20 09:23:00,2026-04-20 09:23:00,111.35,112.30,109.60,112.30,SELL,111.35,112.35,108.00,109.40,2026-04-20 09:23:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY
56,2026-04-20 09:26:00,2026-04-20 09:26:00,99.75,102.45,99.30,101.95,BUY,99.75,110.65,99.30,110.30,2026-04-20 09:26:12,BUY,0.029936,NO_SIGNAL,0.432347,NO_SIGNAL,0.035113,BUY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1806,2026-04-20 15:19:00,2026-04-20 15:19:00,136.10,139.30,135.95,138.00,BUY,136.10,139.70,132.50,137.35,2026-04-20 15:19:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY
1816,2026-04-20 15:21:00,2026-04-20 15:21:00,130.70,134.50,130.70,134.05,BUY,130.70,136.50,130.70,135.40,2026-04-20 15:21:12,BUY,-0.011232,NO_SIGNAL,0.987715,BUY,-0.015669,BUY
1826,2026-04-20 15:23:00,2026-04-20 15:23:00,133.70,135.65,133.50,134.65,BUY,133.70,135.75,132.90,134.85,2026-04-20 15:23:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY
1831,2026-04-20 15:24:00,2026-04-20 15:24:00,134.70,135.70,134.00,135.70,BUY,134.70,136.30,132.80,135.55,2026-04-20 15:24:12,BUY,NaN,NaN,NaN,NaN,NaN,BUY


In [99]:
pwd

'D:\\Study\\Programs\\trading'